# M2: 고CLV 조건부 상품 반응 서브스페이스 (Dunnhumby, seed 42)

기본 LightGCN 64차원은 그대로 두고, 고CLV 고객에게만 활성화되는 8차원 상품 반응 공간을 함께 학습합니다. 보조 사용자 layer-0은 정확히 0이므로 자유 사용자 보조 임베딩이 없고, 구매이력의 상품 반응이 그래프를 통해 모여 사용자 보조표현을 만듭니다. 동일 초기화 `rho=0`, 실제 고CLV gate, 사용자 degree 구간 안에서 gate를 순열한 대조군을 모두 100 epoch로 학습합니다. 최종 test와 holdout은 생성하지 않습니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

REPO_URL = 'https://github.com/jung-un/clv-m2-lightgcn-runner.git'
SOURCE_COMMIT = '355e9c559888e83b22acf761d99469cb0b2b068b'
REPO_DIR = '/content/clv-m2-lightgcn-runner'
if len(SOURCE_COMMIT) != 40:
    raise RuntimeError('검토된 소스 커밋을 SOURCE_COMMIT에 고정해야 합니다')
!if [ -d {REPO_DIR}/.git ]; then git -C {REPO_DIR} fetch origin; else git clone {REPO_URL} {REPO_DIR}; fi
!git -C {REPO_DIR} checkout {SOURCE_COMMIT}
%cd {REPO_DIR}


In [ ]:
import json
import torch
from lightgcn_clv_high_clv_item_response import (
    configure_high_clv_item_response_run,
    preflight_summary,
    run_high_clv_item_response_screen,
)

assert torch.cuda.is_available(), '런타임 유형에서 GPU를 선택하세요.'
cfg = configure_high_clv_item_response_run()
print(json.dumps(preflight_summary(cfg), ensure_ascii=False, indent=2))


In [ ]:
result_df = run_high_clv_item_response_screen(cfg)


In [ ]:
from IPython.display import display
import pandas as pd

print('1) 절대지표: M1, rho=0, 실제 고CLV gate, degree-matched gate shuffle, ID-only')
display(result_df)
print('2) 대조군별 전체·CLV 구간 성과')
display(pd.DataFrame(result_df.attrs['comparison']))
print('3) rho=0 대비 CLV 구간별 Top-10 변경')
display(pd.DataFrame(result_df.attrs['top10_overlap']))
print('4) 공동학습 ID-only 대비 CLV 구간별 Top-10 변경')
display(pd.DataFrame(result_df.attrs['direct_overlap']))
print('5) 실제 gate 대 degree-matched shuffle Top-10 변경')
display(pd.DataFrame(result_df.attrs['attribution_overlap']))
print('6) 보조점수의 실제 영향력')
display(pd.DataFrame(result_df.attrs['score_diagnostics']))
print('7) 사전 판정 규칙 결과')
print(json.dumps(result_df.attrs['screening_reading'], ensure_ascii=False, indent=2))
print('8) 저장 파일')
print(json.dumps(result_df.attrs['result_paths'], ensure_ascii=False, indent=2))
